# Flash Floods KY Python Scripts

## Script #1: Moon & Sun Calculations

In [2]:
# Import Libraries and Dependencies
from datetime import datetime
from pathlib import Path
import math
import ephem
import pandas as pd

In [7]:
# Read dataset into dataframe, then inspect it
df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

df.head()

,event_id,county_name,begin_location,begin_date,begin_time,begin_range,end_range,end_location,end_date,end_time,begin_lat,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,1.48,1.28,SMYRNA,2015-04-03,348,38.1500,-85.6600,38.1536,-85.6547,2015,0.0,0.0
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,1.29,1.47,NEWBURG,2015-04-03,354,38.1600,-85.7000,38.1633,-85.6969,2015,0.0,0.0
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0.88,1.09,LAWRENCEBURG,2015-04-03,400,38.0229,-84.8866,38.0243,-84.8836,2015,0.0,0.0
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0.23,0.05,ELK CREEK,2015-04-03,400,38.0994,-85.3742,38.1038,-85.3727,2015,0.0,0.0
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0.00,0.21,LOUISVILLE,2015-04-03,400,38.2300,-85.7800,38.2296,-85.7792,2015,0.0,0.0


In [8]:
# Create a function that calculates sunrise or sunset for an observer
def get_sun_event(observer, sun, event, never_message):
    """Calculate sunrise or sunset for an observer."""
    
    try:
        result = event(sun)
        return result.datetime().strftime("%Y-%m-%d %H:%M:%S")
    
    except (ephem.CircumpolarError, ephem.AlwaysUpError):
        return "Sun always up"
    
    except ephem.NeverUpError:
        return never_message

In [9]:
# Create a function that determines the moon's phase from illumination and lunar age
def determine_moon_phase(illumination, lunation_age):
    """Determine the moon's phase from illumination and lunar age."""
    
    if illumination < 0.03:
        return "New Moon"
    
    if illumination > 0.97:
        return "Full Moon"
    
    if lunation_age < 14.77:
        if illumination < 0.45:
            return "Waxing Crescent"
        if illumination < 0.55:
            return "First Quarter"
        return "Waxing Gibbous"
    
    if illumination > 0.55:
        return "Waning Gibbous"
    
    if illumination > 0.45:
        return "Third Quarter"
    
    return "Waning Crescent"

In [10]:
# Create a function that calculates local sun and moon data for one row before applying it to the entire dataframe
def calculate_local_astro_data(row: pd.Series) -> pd.Series:
    """Calculate local sun and moon data for one row."""
    
    try:
        # Get event date
        date_obj = datetime.strptime(
            row["begin_date"],
            "%Y-%m-%d"
        )
        
        # Create observer using event coordinates
        observer = ephem.Observer()
        observer.lat = str(row["begin_lat"])
        observer.lon = str(row["begin_lon"])
        observer.date = date_obj
        
        # --------------------------------------------------
        # SUN CALCULATIONS
        # --------------------------------------------------
        
        sun = ephem.Sun()
        sun.compute(observer)
        
        sun_altitude = math.degrees(sun.alt)
        sun_azimuth = math.degrees(sun.az)
        
        # Calculate sunrise and sunset
        observer.date = date_obj.date()
        
        sunrise = get_sun_event(
            observer,
            sun,
            observer.next_rising,
            "Sun never rises"
        )
        
        sunset = get_sun_event(
            observer,
            sun,
            observer.next_setting,
            "Sun never sets"
        )
        
        # --------------------------------------------------
        # MOON CALCULATIONS
        # --------------------------------------------------
        
        observer.date = date_obj
        
        moon = ephem.Moon()
        moon.compute(observer)
        
        moon_altitude = math.degrees(moon.alt)
        moon_azimuth = math.degrees(moon.az)
        
        # Calculate moon illumination
        illumination = moon.phase / 100
        
        # Calculate lunar age
        previous_new_moon = ephem.previous_new_moon(observer.date)
        next_new_moon = ephem.next_new_moon(observer.date)
        
        lunation_age = (
            (observer.date - previous_new_moon)
            * 29.53
            / (next_new_moon - previous_new_moon)
        )
        
        # Determine moon phase
        moon_phase = determine_moon_phase(
            illumination,
            lunation_age
        )
        
        # Return results
        return pd.Series({
            "sun_altitude_deg": round(sun_altitude, 2),
            "sun_azimuth_deg": round(sun_azimuth, 2),
            "sunrise_utc": sunrise,
            "sunset_utc": sunset,
            "moon_altitude_deg": round(moon_altitude, 2),
            "moon_azimuth_deg": round(moon_azimuth, 2),
            "moon_phase_name": moon_phase,
            "moon_illumination_pct": round(illumination * 100, 2)
        })
        
    except Exception as e:
        return pd.Series({
            "sun_altitude_deg": None,
            "sun_azimuth_deg": None,
            "sunrise_utc": f"Error: {e}",
            "sunset_utc": f"Error: {e}",
            "moon_altitude_deg": None,
            "moon_azimuth_deg": None,
            "moon_phase_name": f"Error: {e}",
            "moon_illumination_pct": None
        })

In [11]:
# Test calculate_local_astro_data function with the first row of the dataframe
test_result = calculate_local_astro_data(df.iloc[0])

test_result

sun_altitude_deg                        0.85
sun_azimuth_deg                       276.15
sunrise_utc              2015-04-03 11:24:48
sunset_utc               2015-04-03 00:06:46
moon_altitude_deg                      14.25
moon_azimuth_deg                      101.85
moon_phase_name                    Full Moon
moon_illumination_pct                  97.95
dtype: object

In [12]:
# Run astronomical calculations and display results
astro_data = df.apply(calculate_local_astro_data, axis=1)

astro_data.head()

,sun_altitude_deg,sun_azimuth_deg,sunrise_utc,sunset_utc,moon_altitude_deg,moon_azimuth_deg,moon_phase_name,moon_illumination_pct
0,0.85,276.15,2015-04-03 11:24:48,2015-04-03 00:06:46,14.25,101.85,Full Moon,97.95
1,0.88,276.12,2015-04-03 11:24:57,2015-04-03 00:06:56,14.21,101.82,Full Moon,97.95
2,0.31,276.63,2015-04-03 11:21:47,2015-04-03 00:03:35,14.87,102.33,Full Moon,97.95
3,0.65,276.33,2015-04-03 11:23:41,2015-04-03 00:05:35,14.48,102.02,Full Moon,97.95
4,0.94,276.07,2015-04-03 11:25:13,2015-04-03 00:07:18,14.14,101.79,Full Moon,97.95


In [13]:
# Combine event_id with the calculated astronomy data
output_df = pd.concat(
    [df[["event_id"]], astro_data],
    axis=1
)

output_df.tail()

,event_id,sun_altitude_deg,sun_azimuth_deg,sunrise_utc,sunset_utc,moon_altitude_deg,moon_azimuth_deg,moon_phase_name,moon_illumination_pct
1752,1297632,-9.38,270.33,2025-10-07 11:42:53,2025-10-07 23:15:17,11.36,91.22,Full Moon,99.93
1753,1297629,-9.13,270.14,2025-10-07 11:44:10,2025-10-07 23:16:32,11.12,91.04,Full Moon,99.93
1754,1295858,-10.04,270.83,2025-10-07 11:39:30,2025-10-07 23:12:01,12.02,91.72,Full Moon,99.93
1755,1295859,-10.02,270.84,2025-10-07 11:39:36,2025-10-07 23:12:00,12.01,91.74,Full Moon,99.93
1756,1295860,-10.06,270.86,2025-10-07 11:39:23,2025-10-07 23:11:50,12.05,91.76,Full Moon,99.93


In [14]:
# Save dataframe as CSV file
output_df.to_csv("../data/processed/flash_floods_ky_moon_sun_data.csv", index=False)

## Script #2: Chi-Square Goodness of Fit

In [15]:
# Import Libraries and Dependencies
from pathlib import Path
import pandas as pd
from scipy.stats import chisquare

In [16]:
# Read dataset into dataframe
df = pd.read_csv("../data/processed/flash_floods_ky_moon_sun_data.csv")

df.head()

,event_id,sun_altitude_deg,sun_azimuth_deg,sunrise_utc,sunset_utc,moon_altitude_deg,moon_azimuth_deg,moon_phase_name,moon_illumination_pct
0,564702,0.85,276.15,2015-04-03 11:24:48,2015-04-03 00:06:46,14.25,101.85,Full Moon,97.95
1,564703,0.88,276.12,2015-04-03 11:24:57,2015-04-03 00:06:56,14.21,101.82,Full Moon,97.95
2,564704,0.31,276.63,2015-04-03 11:21:47,2015-04-03 00:03:35,14.87,102.33,Full Moon,97.95
3,564706,0.65,276.33,2015-04-03 11:23:41,2015-04-03 00:05:35,14.48,102.02,Full Moon,97.95
4,564705,0.94,276.07,2015-04-03 11:25:13,2015-04-03 00:07:18,14.14,101.79,Full Moon,97.95


In [17]:
# Create bin boundaries 
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

In [18]:
# Create bin labels
labels = [
    "0-10%", 
    "10-20%", 
    "20-30%", 
    "30-40%", 
    "40-50%", 
    "50-60%", 
    "60-70%", 
    "70-80%", 
    "80-90%", 
    "90-100%"
]

In [19]:
# Assign each flash flooding event to a moon illumination bin
df["illumination_bin"] = pd.cut(
    df["moon_illumination_pct"], 
    bins=bins, 
    labels=labels, 
    include_lowest=True
)

df[["moon_illumination_pct", "illumination_bin"]].head(20)

,moon_illumination_pct,illumination_bin
0,97.95,90-100%
1,97.95,90-100%
2,97.95,90-100%
3,97.95,90-100%
4,97.95,90-100%
5,97.95,90-100%
6,97.95,90-100%
7,97.95,90-100%
8,97.95,90-100%
9,97.95,90-100%


In [20]:
# Count observed flash flooding events
observed = df["illumination_bin"].value_counts().sort_index()

observed

illumination_bin
0-10%      453
10-20%     121
20-30%     108
30-40%      92
40-50%     114
50-60%     125
60-70%      67
70-80%      62
80-90%     192
90-100%    423
Name: count, dtype: int64

In [21]:
# Calculate expected flash flooding event counts
expected = [len(df) / len(observed)] * len(observed)

expected

[175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7]

In [22]:
# Run Chi-Square Goodness of Fit test and display the results
chi2, p_value = chisquare(
    f_obs=observed, 
    f_exp=expected
)

print("Observed Counts:")
print(observed)

print("\nExpected Counts:")
print(expected)

print(f"\nChi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_value:.6f}")

Observed Counts:
illumination_bin
0-10%      453
10-20%     121
20-30%     108
30-40%      92
40-50%     114
50-60%     125
60-70%      67
70-80%      62
80-90%     192
90-100%    423
Name: count, dtype: int64

Expected Counts:
[175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7, 175.7]

Chi-Square Statistic: 1047.3540
P-Value: 0.000000


## Script #3: Oceanic Nino Index 

In [23]:
# Import Libraries and Dependencies
import io
from pathlib import Path
import numpy as np
import pandas as pd

In [24]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

flood_df.head()

,event_id,county_name,begin_location,begin_date,begin_time,begin_range,end_range,end_location,end_date,end_time,begin_lat,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,1.48,1.28,SMYRNA,2015-04-03,348,38.1500,-85.6600,38.1536,-85.6547,2015,0.0,0.0
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,1.29,1.47,NEWBURG,2015-04-03,354,38.1600,-85.7000,38.1633,-85.6969,2015,0.0,0.0
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0.88,1.09,LAWRENCEBURG,2015-04-03,400,38.0229,-84.8866,38.0243,-84.8836,2015,0.0,0.0
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0.23,0.05,ELK CREEK,2015-04-03,400,38.0994,-85.3742,38.1038,-85.3727,2015,0.0,0.0
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0.00,0.21,LOUISVILLE,2015-04-03,400,38.2300,-85.7800,38.2296,-85.7792,2015,0.0,0.0


In [25]:
# Read the ONI text file
with open("../data/raw/oni_backup.txt", "r", encoding="utf-8") as f: oni_text = f.read()

print(oni_text[:1000])

 SEAS  YR   TOTAL   ANOM
  DJF 1950  24.72  -1.53
  JFM 1950  25.17  -1.34
  FMA 1950  25.75  -1.16
  MAM 1950  26.12  -1.18
  AMJ 1950  26.32  -1.07
  MJJ 1950  26.31  -0.85
  JJA 1950  26.21  -0.54
  JAS 1950  25.96  -0.42
  ASO 1950  25.76  -0.39
  SON 1950  25.63  -0.44
  OND 1950  25.48  -0.60
  NDJ 1950  25.34  -0.80
  DJF 1951  25.42  -0.82
  JFM 1951  25.96  -0.54
  FMA 1951  26.74  -0.17
  MAM 1951  27.48   0.18
  AMJ 1951  27.75   0.36
  MJJ 1951  27.75   0.58
  JJA 1951  27.44   0.70
  JAS 1951  27.28   0.89
  ASO 1951  27.14   0.99
  SON 1951  27.22   1.15
  OND 1951  27.12   1.04
  NDJ 1951  26.95   0.81
  DJF 1952  26.78   0.53
  JFM 1952  26.87   0.37
  FMA 1952  27.25   0.34
  MAM 1952  27.60   0.29
  AMJ 1952  27.59   0.20
  MJJ 1952  27.17   0.00
  JJA 1952  26.67  -0.08
  JAS 1952  26.39   0.00
  ASO 1952  26.30   0.15
  SON 1952  26.17   0.10
  OND 1952  26.13   0.04
  NDJ 1952  26.29   0.15
  DJF 1953  26.65   0.40
  JFM 1953  27.10   0.60
  FMA 1953  27.53   0.63


In [26]:
# Parse ONI text into a dataframe, then check results
oni_df = pd.read_csv(
    io.StringIO(oni_text.strip()),
    sep=r"\s+",
    dtype={"YR": int, "ANOM": float}
)

oni_df.head()

,SEAS,YR,TOTAL,ANOM
0,DJF,1950,24.72,-1.53
1,JFM,1950,25.17,-1.34
2,FMA,1950,25.75,-1.16
3,MAM,1950,26.12,-1.18
4,AMJ,1950,26.32,-1.07


In [27]:
# Map ONI seasons to a central calendar month, then check results
season_to_month = {
    "DJF": 1,
    "JFM": 2,
    "FMA": 3,
    "MAM": 4,
    "AMJ": 5,
    "MJJ": 6,
    "JJA": 7,
    "JAS": 8,
    "ASO": 9,
    "SON": 10,
    "OND": 11,
    "NDJ": 12,
}

oni_df["month"] = oni_df["SEAS"].map(season_to_month)

oni_df.head()

,SEAS,YR,TOTAL,ANOM,month
0,DJF,1950,24.72,-1.53,1
1,JFM,1950,25.17,-1.34,2
2,FMA,1950,25.75,-1.16,3
3,MAM,1950,26.12,-1.18,4
4,AMJ,1950,26.32,-1.07,5


In [28]:
# Create a date column using the year and month, then check results
oni_df["date"] = pd.to_datetime(
    dict(
        year=oni_df["YR"],
        month=oni_df["month"],
        day=1
    )
)

oni_df.head()

,SEAS,YR,TOTAL,ANOM,month,date
0,DJF,1950,24.72,-1.53,1,1950-01-01
1,JFM,1950,25.17,-1.34,2,1950-02-01
2,FMA,1950,25.75,-1.16,3,1950-03-01
3,MAM,1950,26.12,-1.18,4,1950-04-01
4,AMJ,1950,26.32,-1.07,5,1950-05-01


In [29]:
# Classify ENSO phase based on ONI anomaly, then check results
oni_df["enso_phase"] = np.select(
    [
        oni_df["ANOM"] >= 0.5,
        oni_df["ANOM"] <= -0.5
    ],
    [
        "El Nino",
        "La Nina"
    ],
    default="Neutral"
)

oni_df.head()

,SEAS,YR,TOTAL,ANOM,month,date,enso_phase
0,DJF,1950,24.72,-1.53,1,1950-01-01,La Nina
1,JFM,1950,25.17,-1.34,2,1950-02-01,La Nina
2,FMA,1950,25.75,-1.16,3,1950-03-01,La Nina
3,MAM,1950,26.12,-1.18,4,1950-04-01,La Nina
4,AMJ,1950,26.32,-1.07,5,1950-05-01,La Nina


In [30]:
# Select the columns needed for the analysis and rename them for clarity, then check results
oni_df = oni_df[
    ["date", "SEAS", "ANOM", "enso_phase"]
].rename(
    columns={
        "SEAS": "oni_season",
        "ANOM": "oni_anomaly"
    }
)

oni_df.head()

,date,oni_season,oni_anomaly,enso_phase
0,1950-01-01,DJF,-1.53,La Nina
1,1950-02-01,JFM,-1.34,La Nina
2,1950-03-01,FMA,-1.16,La Nina
3,1950-04-01,MAM,-1.18,La Nina
4,1950-05-01,AMJ,-1.07,La Nina


In [31]:
# Sort ONI data chronologically, then check results
oni_df = oni_df.sort_values("date").reset_index(drop=True)

oni_df.head()

,date,oni_season,oni_anomaly,enso_phase
0,1950-01-01,DJF,-1.53,La Nina
1,1950-02-01,JFM,-1.34,La Nina
2,1950-03-01,FMA,-1.16,La Nina
3,1950-04-01,MAM,-1.18,La Nina
4,1950-05-01,AMJ,-1.07,La Nina


In [32]:
# Sort flash flood data chronologically, then check results
flood_df = flood_df.sort_values("begin_date").reset_index(drop=True)

flood_df[["event_id", "begin_date"]].head()

,event_id,begin_date
0,564702,2015-04-03
1,564703,2015-04-03
2,564704,2015-04-03
3,564706,2015-04-03
4,564705,2015-04-03


In [33]:
# Check the data types of the columns before merging into one dataframe
print("flood_df begin_date:", flood_df["begin_date"].dtype)
print("oni_df date:", oni_df["date"].dtype)

flood_df begin_date: str
oni_df date: datetime64[us]


In [34]:
# Change flood_df begin_date from str datatype to datetime datatype
flood_df["begin_date"] = pd.to_datetime(
    flood_df["begin_date"]
)

print(flood_df["begin_date"].dtype)

datetime64[us]


In [35]:
# Merge each flash flood event with the most recent ONI record occurring on or before the event date, then check results
enriched_df = pd.merge_asof(
    flood_df,
    oni_df,
    left_on="begin_date",
    right_on="date",
    direction="backward"
)

enriched_df.head()

,event_id,county_name,begin_location,begin_date,begin_time,begin_range,end_range,end_location,end_date,end_time,...,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours,date,oni_season,oni_anomaly,enso_phase
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,1.48,1.28,SMYRNA,2015-04-03,348,...,-85.6600,38.1536,-85.6547,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,1.29,1.47,NEWBURG,2015-04-03,354,...,-85.7000,38.1633,-85.6969,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0.88,1.09,LAWRENCEBURG,2015-04-03,400,...,-84.8866,38.0243,-84.8836,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0.23,0.05,ELK CREEK,2015-04-03,400,...,-85.3742,38.1038,-85.3727,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0.00,0.21,LOUISVILLE,2015-04-03,400,...,-85.7800,38.2296,-85.7792,2015,0.0,0.0,2015-04-01,MAM,0.81,El Nino


In [36]:
# Keep event_id and the ONI columns
enriched_df = enriched_df[
    [
        "event_id",
        "oni_season",
        "oni_anomaly",
        "enso_phase"
    ]
]

enriched_df.head()

,event_id,oni_season,oni_anomaly,enso_phase
0,564702,MAM,0.81,El Nino
1,564703,MAM,0.81,El Nino
2,564704,MAM,0.81,El Nino
3,564706,MAM,0.81,El Nino
4,564705,MAM,0.81,El Nino


In [37]:
# Save the merged DataFrame as a CSV file
enriched_df.to_csv("../data/processed/flash_floods_ky_oni_data.csv", index=False)

## Script #4: NLCD Landcover API

In [38]:
# Import Libraries and Dependencies 
import os
from pathlib import Path
import pandas as pd
from arcgis.gis import GIS
from arcgis.raster import ImageryLayer
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

In [39]:
# Load environment variables
load_dotenv()

True

In [40]:
# Add configuration
API_KEY = os.environ.get("arcgis_api_key")
ARCGIS_URL = "https://arcgis.com"

In [41]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

flood_df.head()

,event_id,county_name,begin_location,begin_date,begin_time,begin_range,end_range,end_location,end_date,end_time,begin_lat,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,1.48,1.28,SMYRNA,2015-04-03,348,38.1500,-85.6600,38.1536,-85.6547,2015,0.0,0.0
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,1.29,1.47,NEWBURG,2015-04-03,354,38.1600,-85.7000,38.1633,-85.6969,2015,0.0,0.0
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0.88,1.09,LAWRENCEBURG,2015-04-03,400,38.0229,-84.8866,38.0243,-84.8836,2015,0.0,0.0
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0.23,0.05,ELK CREEK,2015-04-03,400,38.0994,-85.3742,38.1038,-85.3727,2015,0.0,0.0
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0.00,0.21,LOUISVILLE,2015-04-03,400,38.2300,-85.7800,38.2296,-85.7792,2015,0.0,0.0


In [42]:
# Define NLCD Layer IDs
LAYER_IDS = {
    "nlcd":"32e2ccc6416746a9a72b4d216813f84f",
    "elev":"58a541efc59545e6b7137f961d7de883",
    "imperv":"6df535f263dd44f489365eed49461a38",
}

In [43]:
# Define NLCD classes
NLCD_CLASSES = {
    11: "Open Water",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}

In [44]:
# Connect to ArcGIS
def initialize_layers(api_key: str) -> tuple:
    """Connects to the ArcGIS GIS API with strict SSL validation
    and returns the three imagery layers."""

    print("Connecting to ArcGIS Living Atlas layers...")

    gis = GIS(
        ARCGIS_URL,
        api_key=api_key,
        verify_cert=True
    )

    layers = {
        name: ImageryLayer(
            gis.content.get(item_id).url,
            gis=gis
        )
        for name, item_id in LAYER_IDS.items()
    }

    print("All layers connected successfully.")

    return (
        layers["nlcd"],
        layers["elev"],
        layers["imperv"]
    )

In [45]:
# Query One Imagery Layer
def query_layer_value(layer: ImageryLayer, geom: dict):
    """Queries an imagery layer at a point geometry
    and returns its pixel value, or None on failure."""

    try:
        response = layer.identify(
            geometry=geom,
            return_pixel_values=True
        )

        return response.get("value", None)

    except Exception:
        return None

In [46]:
# Process One Flash Flood Event
def process_single_row(idx, row, layers):
    """Worker function to process a single row's
    spatial variables concurrently."""

    nlcd_layer, elev_layer, imp_layer = layers

    mid_lat = (row["begin_lat"] + row["end_lat"]) / 2
    mid_lon = (row["begin_lon"] + row["end_lon"]) / 2

    if pd.isna(mid_lat) or pd.isna(mid_lon):
        return idx, (None, None, None, None)

    geom = {
        "x": mid_lon,
        "y": mid_lat,
        "spatialReference": {"wkid": 4326}
    }

    val_nlcd = query_layer_value(nlcd_layer, geom)
    val_elev = query_layer_value(elev_layer, geom)
    val_imp = query_layer_value(imp_layer, geom)

    c_code = int(val_nlcd) if val_nlcd is not None else None

    c_class = (
        NLCD_CLASSES.get(c_code, "Unknown")
        if c_code is not None
        else None
    )

    elevation = (
        float(val_elev)
        if val_elev is not None
        else None
    )

    impervious = (
        int(val_imp)
        if val_imp is not None
        else None
    )

    return idx, (
        c_code,
        c_class,
        elevation,
        impervious
    )

In [47]:
# Process all flash flooding events within the dataframe
def fetch_geospatial_attributes(
    df: pd.DataFrame,
    layers: tuple,
    max_workers: int = 20,
) -> pd.DataFrame:

    """Fetches geospatial attributes for each row
    and joins them onto the dataframe."""

    print(
        f"Starting batch execution using "
        f"{max_workers} concurrent threads..."
    )

    cols = [
        "nlcd_code",
        "nlcd_class",
        "elevation_m",
        "impervious_surface_pct"
    ]

    results = {}

    total_rows = len(df)

    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        futures = {
            executor.submit(
                process_single_row,
                idx,
                row,
                layers
            ): idx

            for idx, row in df.iterrows()
        }

        for completed, future in enumerate(
            as_completed(futures), 1
        ):

            idx, values = future.result()

            results[idx] = values

            if completed % 100 == 0 or completed == total_rows:
                print(
                    f"Progress: {completed}/{total_rows} "
                    f"rows extracted "
                    f"({completed/total_rows*100:.1f}%)"
                )

    result_df = pd.DataFrame.from_dict(
        results,
        orient="index",
        columns=cols
    )

    return df.join(result_df)

In [48]:
# Connect to ArcGIS
atlas_layers = initialize_layers(API_KEY)

Connecting to ArcGIS Living Atlas layers...
All layers connected successfully.


In [49]:
# Run geospatial extraction
processed_df = fetch_geospatial_attributes(
    flood_df,
    atlas_layers,
    max_workers=25
)

Starting batch execution using 25 concurrent threads...
Progress: 100/1757 rows extracted (5.7%)
Progress: 200/1757 rows extracted (11.4%)
Progress: 300/1757 rows extracted (17.1%)
Progress: 400/1757 rows extracted (22.8%)
Progress: 500/1757 rows extracted (28.5%)
Progress: 600/1757 rows extracted (34.1%)
Progress: 700/1757 rows extracted (39.8%)
Progress: 800/1757 rows extracted (45.5%)
Progress: 900/1757 rows extracted (51.2%)
Progress: 1000/1757 rows extracted (56.9%)
Progress: 1100/1757 rows extracted (62.6%)
Progress: 1200/1757 rows extracted (68.3%)
Progress: 1300/1757 rows extracted (74.0%)
Progress: 1400/1757 rows extracted (79.7%)
Progress: 1500/1757 rows extracted (85.4%)
Progress: 1600/1757 rows extracted (91.1%)
Progress: 1700/1757 rows extracted (96.8%)
Progress: 1757/1757 rows extracted (100.0%)


In [50]:
# Inspect processed dataset
processed_df.head()

,event_id,county_name,begin_location,begin_date,begin_time,begin_range,end_range,end_location,end_date,end_time,...,begin_lon,end_lat,end_lon,year,duration_minutes,duration_hours,nlcd_code,nlcd_class,elevation_m,impervious_surface_pct
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,1.48,1.28,SMYRNA,2015-04-03,348,...,-85.6600,38.1536,-85.6547,2015,0.0,0.0,24,"Developed, High Intensity",149.975,83
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,1.29,1.47,NEWBURG,2015-04-03,354,...,-85.7000,38.1633,-85.6969,2015,0.0,0.0,24,"Developed, High Intensity",140.396,84
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,0.88,1.09,LAWRENCEBURG,2015-04-03,400,...,-84.8866,38.0243,-84.8836,2015,0.0,0.0,21,"Developed, Open Space",236.400,0
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,0.23,0.05,ELK CREEK,2015-04-03,400,...,-85.3742,38.1038,-85.3727,2015,0.0,0.0,21,"Developed, Open Space",209.800,0
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,0.00,0.21,LOUISVILLE,2015-04-03,400,...,-85.7800,38.2296,-85.7792,2015,0.0,0.0,24,"Developed, High Intensity",142.241,88


In [ ]:
# Keep only event_id and NLCD columns
processed_df = processed_df[
    [
        "event_id",
        "nlcd_code",
        "nlcd_class",
        "elevation_m",
        "impervious_surface_pct"
    ]
]

processed_df.head()

,event_id,nlcd_code,nlcd_class,elevation_m,impervious_surface_pct
0,564702,24,"Developed, High Intensity",149.975,83
1,564703,24,"Developed, High Intensity",140.396,84
2,564704,21,"Developed, Open Space",236.400,0
3,564706,21,"Developed, Open Space",209.800,0
4,564705,24,"Developed, High Intensity",142.241,88


In [52]:
# Save processed dataset as CSV file
processed_df.to_csv("../data/processed/flash_floods_ky_nlcd_data.csv", index=False)

## Script #5: Weather API

In [53]:
# Import Libraries and Dependencies
import pandas as pd
import requests
import json
import time
from dotenv import load_dotenv
import os

In [54]:
load_dotenv()

True

In [55]:
weather_api_key = os.getenv("weather_api_key")

In [57]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_2015_2025_cleaned.csv")

flood_df.head()

,event_id,county_name,begin_location,begin_date,begin_time,event_type,deaths_direct,injuries_direct,damage_property_num,damage_crops_num,...,end_location,end_date,end_time,begin_lat,begin_lon,end_lat,end_lon,event_narrative,episode_narrative,year
0,564702,JEFFERSON CO.,SMYRNA,2015-04-03,348,Flash Flood,0,0,0,0,...,SMYRNA,2015-04-03,348,38.1500,-85.6600,38.1536,-85.6547,Law enforcement reported several water rescues...,A stalled frontal boundary across the area bro...,2015
1,564703,JEFFERSON CO.,NEWBURG,2015-04-03,354,Flash Flood,0,0,20000,0,...,NEWBURG,2015-04-03,354,38.1600,-85.7000,38.1633,-85.6969,Law enforcement reported evacuations at 6111 G...,A stalled frontal boundary across the area bro...,2015
2,564704,ANDERSON CO.,LAWRENCEBURG,2015-04-03,400,Flash Flood,0,0,30000,0,...,LAWRENCEBURG,2015-04-03,400,38.0229,-84.8866,38.0243,-84.8836,Emergency Management reported several roads cl...,A stalled frontal boundary across the area bro...,2015
3,564706,SPENCER CO.,ELK CREEK,2015-04-03,400,Flash Flood,0,0,30000,0,...,ELK CREEK,2015-04-03,400,38.0994,-85.3742,38.1038,-85.3727,State officials reported numerous water rescue...,A stalled frontal boundary across the area bro...,2015
4,564705,JEFFERSON CO.,LOUISVILLE,2015-04-03,400,Flash Flood,0,0,0,0,...,LOUISVILLE,2015-04-03,400,38.2300,-85.7800,38.2296,-85.7792,Law enforcement reported a road closure near 1...,A stalled frontal boundary across the area bro...,2015


In [58]:
# Prepare a dataframe for weather extraction
weather_tasks = df[
    [
        "event_id",
        "begin_date",
        "begin_lat",
        "begin_lon",
        "begin_location"
    ]
].copy()

weather_tasks = weather_tasks.dropna(
    subset=[
        "event_id",
        "begin_date",
        "begin_lat",
        "begin_lon"
    ]
).copy()

print(f"Records ready for weather extraction: {len(weather_tasks)}")

Records ready for weather extraction: 1757


In [59]:
# Define weather fields to extract
WEATHER_FIELDS = [
    "maxtemp_f",
    "mintemp_f",
    "avgtemp_f",
    "maxwind_mph",
    "totalprecip_in",
    "avgvis_miles",
    "avghumidity",
    "uv",
    "daily_will_it_rain",
    "daily_chance_of_rain",
    "condition_text"
]

In [60]:
# Create a function to get historical weather data from WeatherAPI
def get_historical_weather(
    event_id,
    event_date,
    latitude,
    longitude,
    api_key,
    max_retries=3
):
    
    url = "https://api.weatherapi.com/v1/history.json"
    
    params = {
        "key": api_key,
        "q": f"{latitude},{longitude}",
        "dt": event_date
    }
    
    for attempt in range(max_retries):
        
        try:
            response = requests.get(
                url,
                params=params,
                timeout=60
            )
            
            response.raise_for_status()
            
            data = response.json()
            
            day_data = data["forecast"]["forecastday"][0]["day"]
            location_data = data["location"]
            condition_data = day_data.get("condition", {})
            
            result = {
                "event_id": event_id,
                "date": event_date,
                "latitude": latitude,
                "longitude": longitude,
                "location": location_data.get("name")
            }
            
            # Extract standard weather fields
            for field in WEATHER_FIELDS:
                
                if field not in [
                    "condition_text"
                ]:
                    result[field] = day_data.get(field)
            
            # Extract nested condition fields
            result["condition_text"] = condition_data.get("text")
            
            return result
            
        except requests.exceptions.RequestException as e:
            
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            
            else:
                return {
                    "event_id": event_id,
                    "date": event_date,
                    "latitude": latitude,
                    "longitude": longitude,
                    "location": None,
                    "error": str(e)
                }

In [ ]:
# Test the get_historical_weather function with the first row of the weather_tasks dataframe
test_row = weather_tasks.iloc[0]

test_result = get_historical_weather(
    event_id=test_row["event_id"],
    event_date=pd.to_datetime(
        test_row["begin_date"]
    ).strftime("%Y-%m-%d"),
    latitude=test_row["begin_lat"],
    longitude=test_row["begin_lon"],
    api_key=weather_api_key
)

test_result

{'event_id': np.int64(564702),
 'date': '2015-04-03',
 'latitude': np.float64(38.15),
 'longitude': np.float64(-85.66),
 'location': 'Newburg',
 'maxtemp_f': 66.6,
 'mintemp_f': 45.3,
 'avgtemp_f': 58.9,
 'maxwind_mph': 13.9,
 'totalprecip_in': 2.19,
 'avgvis_miles': 4.0,
 'avghumidity': 95,
 'uv': 2.7,
 'daily_will_it_rain': 1,
 'daily_chance_of_rain': 100,
 'condition_text': 'Moderate or heavy rain shower'}

In [64]:
# Run weather extraction for all records in the weather_tasks dataframe
weather_results = []

total_records = len(weather_tasks)

print("Starting weather data extraction...")
print(f"Total records: {total_records}")

for i, (_, row) in enumerate(weather_tasks.iterrows(), start=1):
    
    result = get_historical_weather(
        event_id=row["event_id"],
        event_date=pd.to_datetime(
            row["begin_date"]
        ).strftime("%Y-%m-%d"),
        latitude=row["begin_lat"],
        longitude=row["begin_lon"],
        api_key=weather_api_key
    )
    
    weather_results.append(result)
    
    if i % 25 == 0 or i == total_records:
        print(
            f"Processed {i:,} / {total_records:,} "
            f"({i / total_records:.1%})"
        )
    
    # Pause between requests
    time.sleep(0.2)

print("Weather extraction complete!")

Starting weather data extraction...
Total records: 1757
Processed 25 / 1,757 (1.4%)
Processed 50 / 1,757 (2.8%)
Processed 75 / 1,757 (4.3%)
Processed 100 / 1,757 (5.7%)
Processed 125 / 1,757 (7.1%)
Processed 150 / 1,757 (8.5%)
Processed 175 / 1,757 (10.0%)
Processed 200 / 1,757 (11.4%)
Processed 225 / 1,757 (12.8%)
Processed 250 / 1,757 (14.2%)
Processed 275 / 1,757 (15.7%)
Processed 300 / 1,757 (17.1%)
Processed 325 / 1,757 (18.5%)
Processed 350 / 1,757 (19.9%)
Processed 375 / 1,757 (21.3%)
Processed 400 / 1,757 (22.8%)
Processed 425 / 1,757 (24.2%)
Processed 450 / 1,757 (25.6%)
Processed 475 / 1,757 (27.0%)
Processed 500 / 1,757 (28.5%)
Processed 525 / 1,757 (29.9%)
Processed 550 / 1,757 (31.3%)
Processed 575 / 1,757 (32.7%)
Processed 600 / 1,757 (34.1%)
Processed 625 / 1,757 (35.6%)
Processed 650 / 1,757 (37.0%)
Processed 675 / 1,757 (38.4%)
Processed 700 / 1,757 (39.8%)
Processed 725 / 1,757 (41.3%)
Processed 750 / 1,757 (42.7%)
Processed 775 / 1,757 (44.1%)
Processed 800 / 1,757 (

In [65]:
# Create weather dataframe
weather_df = pd.DataFrame(weather_results)

weather_df.head()

,event_id,date,latitude,longitude,location,maxtemp_f,mintemp_f,avgtemp_f,maxwind_mph,totalprecip_in,avgvis_miles,avghumidity,uv,daily_will_it_rain,daily_chance_of_rain,condition_text
0,564702,2015-04-03,38.1500,-85.6600,Newburg,66.6,45.3,58.9,13.9,2.19,4.0,95,2.7,1,100,Moderate or heavy rain shower
1,564703,2015-04-03,38.1600,-85.7000,Poplar Hills,66.6,45.3,58.9,13.9,2.19,4.0,95,2.7,1,100,Moderate or heavy rain shower
2,564704,2015-04-03,38.0229,-84.8866,Stringtown,66.2,45.4,59.1,17.7,2.67,4.0,96,1.3,1,100,Moderate or heavy rain shower
3,564706,2015-04-03,38.0994,-85.3742,Elk Creek,65.5,45.0,58.7,15.4,2.65,3.0,95,1.9,1,100,Moderate or heavy rain shower
4,564705,2015-04-03,38.2300,-85.7800,South Parkland,66.9,44.8,58.2,13.0,2.03,4.0,95,2.7,1,100,Moderate or heavy rain shower


In [66]:
# Drop unnecessary columns and display the first few rows of the weather dataframe
weather_df = weather_df.drop(columns=
    ['date',
    'latitude',
    'longitude',
    'location'
    ])

weather_df.head()

,event_id,maxtemp_f,mintemp_f,avgtemp_f,maxwind_mph,totalprecip_in,avgvis_miles,avghumidity,uv,daily_will_it_rain,daily_chance_of_rain,condition_text
0,564702,66.6,45.3,58.9,13.9,2.19,4.0,95,2.7,1,100,Moderate or heavy rain shower
1,564703,66.6,45.3,58.9,13.9,2.19,4.0,95,2.7,1,100,Moderate or heavy rain shower
2,564704,66.2,45.4,59.1,17.7,2.67,4.0,96,1.3,1,100,Moderate or heavy rain shower
3,564706,65.5,45.0,58.7,15.4,2.65,3.0,95,1.9,1,100,Moderate or heavy rain shower
4,564705,66.9,44.8,58.2,13.0,2.03,4.0,95,2.7,1,100,Moderate or heavy rain shower


In [67]:
# Save dataframe as CSV
weather_df.to_csv("../data/raw/flash_floods_ky_weather_conditions.csv",index=False)